# Melatih Convnet dari Nol pada Data Kecil

**Buku:** *Deep Learning dengan Python* — Bab 5, Deep Learning untuk Visi Komputer

Klasifikasi kucing vs anjing dengan hanya 2.000 citra latih. Kita mulai dari baseline (overfitting berat), lalu menambahkan augmentasi data dan dropout.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- Data diunduh otomatis (arsip Microsoft *Cats vs Dogs*); tidak perlu akun Kaggle atau path lokal.
- `ImageDataGenerator` + `fit_generator` (dihapus di Keras 3) → `image_dataset_from_directory` + `fit`.
- Augmentasi kini berupa lapisan di dalam model (`RandomFlip`, `RandomRotation`, `RandomZoom`).
- Penskalaan piksel dilakukan oleh lapisan `Rescaling` sehingga model menerima piksel mentah.
- Model disimpan dalam format `.keras` (bukan `.h5`) dan dipilih otomatis lewat `ModelCheckpoint`.
- Ditambah evaluasi pada data **uji** (asli hanya melaporkan validasi).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Mengunduh dan menyiapkan subset kecil
1.000 citra latih, 500 validasi, dan 500 uji per kelas.

In [ ]:
import os, zipfile, shutil, pathlib

URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", origin=URL)

asli = pathlib.Path("data/PetImages")
if not asli.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("data")

def berkas_valid(folder):
    """Kembalikan berkas JPEG yang sah (dataset asli memuat beberapa berkas rusak)."""
    hasil = []
    for p in sorted(folder.glob("*.jpg"), key=lambda q: int(q.stem) if q.stem.isdigit() else -1):
        with open(p, "rb") as f:
            if b"JFIF" in f.read(10):
                hasil.append(p)
    return hasil

kecil = pathlib.Path("data/kucing_vs_anjing_kecil")
PEMBAGIAN = {"latih": (0, 1000), "validasi": (1000, 1500), "uji": (1500, 2000)}
if not kecil.exists():
    for kelas_asli, kelas in [("Cat", "kucing"), ("Dog", "anjing")]:
        daftar = berkas_valid(asli / kelas_asli)
        for bagian, (awal, akhir) in PEMBAGIAN.items():
            tujuan = kecil / bagian / kelas
            tujuan.mkdir(parents=True, exist_ok=True)
            for p in daftar[awal:akhir]:
                shutil.copyfile(p, tujuan / p.name)

for bagian in PEMBAGIAN:
    print(bagian, {k: len(list((kecil / bagian / k).glob("*.jpg"))) for k in ["kucing", "anjing"]})

In [ ]:
UKURAN = (150, 150)
ds_latih = keras.utils.image_dataset_from_directory(
    kecil / "latih", image_size=UKURAN, batch_size=32, seed=42)
ds_val = keras.utils.image_dataset_from_directory(
    kecil / "validasi", image_size=UKURAN, batch_size=32, shuffle=False)
ds_uji = keras.utils.image_dataset_from_directory(
    kecil / "uji", image_size=UKURAN, batch_size=32, shuffle=False)
print("Nama kelas:", ds_latih.class_names)   # urutan alfabet: anjing=0, kucing=1

In [ ]:
for citra, label in ds_latih.take(1):
    print("Bentuk batch citra:", citra.shape, "| label:", label.shape)

## 2. Baseline tanpa regularisasi

In [ ]:
from keras import layers

def bangun_convnet(augmentasi=None, dropout=0.0):
    masukan = keras.Input(shape=UKURAN + (3,))
    x = augmentasi(masukan) if augmentasi is not None else masukan
    x = layers.Rescaling(1.0 / 255)(x)
    for filter_ in [32, 64, 128, 128]:
        x = layers.Conv2D(filter_, 3, activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    if dropout:
        x = layers.Dropout(dropout)(x)
    x = layers.Dense(512, activation="relu")(x)
    keluaran = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(masukan, keluaran)
    model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),
                  metrics=["accuracy"])
    return model

model = bangun_convnet()
model.summary()

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, judul=""):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h["accuracy"], "o", ms=3, label="Latih")
    ax[0].plot(epoch, h["val_accuracy"], "-", label="Validasi")
    ax[0].set(xlabel="Epoch", ylabel="Akurasi", title=judul)
    ax[1].plot(epoch, h["loss"], "o", ms=3, label="Latih")
    ax[1].plot(epoch, h["val_loss"], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel="Loss", title=judul)
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

cb = [keras.callbacks.ModelCheckpoint("convnet_dari_nol.keras", save_best_only=True, monitor="val_loss")]
riwayat = model.fit(ds_latih, epochs=30, validation_data=ds_val, callbacks=cb)
plot_riwayat(riwayat, "Baseline tanpa augmentasi")

## 3. Augmentasi data

In [ ]:
augmentasi = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
], name="augmentasi")

import numpy as np
plt.figure(figsize=(8, 8))
for citra, _ in ds_latih.take(1):
    for i in range(9):
        hasil = augmentasi(citra, training=True)
        plt.subplot(3, 3, i + 1)
        plt.imshow(keras.ops.convert_to_numpy(hasil[0]).astype("uint8"))
        plt.axis("off")
plt.show()

In [ ]:
model = bangun_convnet(augmentasi=augmentasi, dropout=0.5)
cb = [keras.callbacks.ModelCheckpoint("convnet_dengan_augmentasi.keras",
                                      save_best_only=True, monitor="val_loss")]
riwayat = model.fit(ds_latih, epochs=100, validation_data=ds_val, callbacks=cb)
plot_riwayat(riwayat, "Dengan augmentasi + dropout")

## 4. Evaluasi pada data uji (sekali, di akhir)

In [ ]:
for nama in ["convnet_dari_nol.keras", "convnet_dengan_augmentasi.keras"]:
    terbaik = keras.models.load_model(nama)
    _, akurasi = terbaik.evaluate(ds_uji, verbose=0)
    print(f"{nama:35s} akurasi uji: {akurasi:.2%}")